In [158]:
import pandas as pd
import random
import time
import numpy as np
from scipy.stats import zscore

In [159]:

# Reemplaza los valores atípicos que pueden generar confusión
def arreglar_dispersion(df, columnas, z_score_umbral):
    for col in columnas:
        # Z-score mide cuántas desviaciones estándar se encuentra un valor respecto a la media.
        z_scores = np.abs(zscore(df[col], nan_policy='omit'))  # Calcula Z-score (da un valor indicando cuánta desviación tiene de la media)
        df.loc[z_scores > z_score_umbral, col] = np.nan  # Sustituye atípicos con NaN (en el caso de que la desviación sea mayor que el umbral)


In [160]:
# Columnas seleccionas del dataset con la información de los juegos
INFO_JUEGO_COLS = ['id',
                   'primary',
                   'yearpublished',
                   'playingtime',
                   'boardgamecategory', 
                   'maxplayers',
                   'average']

# Columans seleccionadas de las reviews
RATING_COLS = ['user',
               'rating',
               'ID', 
               'comment']

PROBABILIDAD_FALLO = 0.2
TOP_JUEGOS = 1000
PORCENTAJE = 0.1
VOTOS_MINIMOS = 10
DELIMITADOR_Z_SCORE = 3
EJEMPLO_USUARIO = "1000rpm"
VALORACION_USUARIO_UMBRAL = 5


FILTRO_CATEGORIA = {
    "boardgamecategory": None
}
FILTRO_NUMERICO = {
    "playingtime": None,
    "average": {"measure": "above", "threshold": 7}
}

random.seed(int(time.time()))

In [161]:
# Lectura CSV
info_juegos = pd.read_csv("../data/games_detailed_info.csv",
        usecols=INFO_JUEGO_COLS, 
        dtype={'id': 'uint32', 
                'primary': 'string', 
                'yearpublished': 'uint16', 
                'maxplayers': 'uint16',
                'playingtime': 'uint32',
                'average': 'float16'
                })


# Solucionar dispersión, sustituye por nan
arreglar_dispersion(info_juegos, ['playingtime', 'maxplayers'], DELIMITADOR_Z_SCORE)

# Eliminar nulos, duplicados.
info_juegos = info_juegos.dropna()
info_juegos = info_juegos.drop_duplicates()

#  Eliminar espacios en blanco, mayúsculas y reduce a dos decimales la valoración media
info_juegos['primary'] = info_juegos['primary'].str.lower()
info_juegos['primary'] = info_juegos['primary'].str.replace(' ', '_')
info_juegos['average'] = info_juegos['average'].apply(lambda x: np.around(x, decimals=2))

In [162]:
reviews = pd.read_csv("../data/bgg-26m-reviews.csv", 
                      usecols=RATING_COLS,
                      dtype={'user': 'string', 
                             'rating': 'float16', 
                             'ID': 'uint32',
                             'comment': 'string[pyarrow]' #  Al usar string[pyarrow] se reduce la memoria significativamente
                             }) 

reviews = reviews.dropna()
reviews = reviews.drop_duplicates(subset=["user", "ID"], keep="first")

reviews['rating'] = reviews['rating'].astype(np.int8)

In [163]:
reviews.head()

,user,rating,comment,ID
2,dougthonus,10,"Currently, this sits on my list as my favorite...",13
3,cypar7,10,"I know it says how many plays, but many, many ...",13
7,hreimer,10,i will never tire of this game.. Awesome,13
11,daredevil,10,This is probably the best game I ever played. ...,13
16,hurkle,10,Fantastic game. Got me hooked on games all ove...,13


Indica el ID de aquellos juegos que quieras para generar un CSV de una escala más reducida.

In [164]:
juegos = [10630, 120, 19948, 17133, 6351]
info_juegos = info_juegos[info_juegos['id'].isin(juegos)]
info_juegos.head(14)

,id,primary,yearpublished,maxplayers,playingtime,boardgamecategory,average
91,10630,memoir_'44,2004,8.0,60.0,"['Miniatures', 'Wargame', 'World War II']",7.56
315,17133,railways_of_the_world,2005,6.0,120.0,"['Trains', 'Transportation', 'Video Game Theme']",7.69
714,120,hoity_toity,1990,6.0,45.0,['Bluffing'],6.52
1263,6351,gulo_gulo,2003,6.0,20.0,"['Action / Dexterity', 'Animals', ""Children's ...",6.83
1853,19948,rum_&_pirates,2006,5.0,60.0,"['Dice', 'Miniatures', 'Pirates']",6.43


In [165]:
mini_reviews_random = reviews[reviews['ID'].isin(info_juegos['id'])]

In [166]:
len(mini_reviews_random)

11850

In [167]:
info_juegos.to_csv("../data/info_juegos_optimizado.csv", index=False)

In [168]:
mini_reviews_random.to_csv("../data/bgg-26m-reviews-mini.csv", index=False)